# Week 3 V1 data pipeline: SkillAlign-RAG

**Pathway:** A (documents), with two branches: course files and HRD Corp IndSF PDFs.

**Runs locally in VS Code, not Colab.** The lecturer accepted code that differs from the lab as long as the
deliverables are submitted. Local execution also keeps internal course files off shared services.

**What this notebook shows:** only counts, hashes and short samples from a generic lecture slide and the IndSF
framework. Prepared course text stays local under `data/processed/` and `data/pipeline_v1/course_documents/`, as the lab
allows for data that cannot be shared. Prepared IndSF records are committed.

The pipeline logic is in [`src/week3_pipeline.py`](../src/week3_pipeline.py), which calls the existing extractors.

## Part A: Start with the data

### A1. Requirements to data needs

| Requirement | Needed information | Source | Constraint |
| :--- | :--- | :--- | :--- |
| FR1 | CLO, Taught and Assessed course documents by course | `data/courses/` | Internal; must stay out of the public repo |
| FR3 / RAI1 | Verbatim text with exact document and page reference | Same files | Every prepared record must keep its page |
| FR7 | IndSF skill source text and references | `data/industry_benchmark/` | Published framework; licence terms not verified |
| RAI3 | Personal data removed from extracted text | Pipeline redaction | Regex only; does not detect names |
| NFR4 | Repeatable updates and change detection | SHA-256 per source and output | Hash shows integrity, not quality or permission |

### A2. Workspace and input verification

In [1]:
import json
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
import os, sys
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.week3_pipeline import (COURSE_KIND, INDSF_KIND, KINDS, PipelineStageError, build_lineage_manifest,
                                check_source_integrity, compare_chunking, compare_ocr, compare_runs,
                                execute_pipeline, extract, sha256_file, verify_manifest)

COURSES_DIR = ROOT / "data/courses"
INDSF_DIR = ROOT / "data/industry_benchmark"
PROCESSED_DIR = ROOT / "data/processed"
MANIFEST = ROOT / "data/week3_manifest.json"
PIPELINE_DIR = ROOT / "data/pipeline_v1"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OPTIONS = {"workers": 6, "ocr_threads": 2}

files = sorted(p for d in (COURSES_DIR, INDSF_DIR) for p in d.rglob("*") if p.is_file())
assert files, "No input files found."
print("Input files:", len(files), "| total MB:", round(sum(p.stat().st_size for p in files) / 1e6, 1))
print("By format:", dict(Counter(p.suffix.lower() for p in files)))
print("By folder:")
for folder, count in sorted(Counter(p.parent.relative_to(ROOT).as_posix() for p in files).items()):
    print(f"  {count:>3}  {folder}")
missing = [d.relative_to(ROOT).as_posix() for d in COURSES_DIR.glob("*/*") if d.is_dir() and not any(d.iterdir())]
print("Empty evidence folders:", missing)

Input files: 40 | total MB: 184.4
By format: {'.pdf': 38, '.pptx': 1, '.docx': 1}
By folder:
    1  data/courses/SAIA1013_Responsible_AI_and_Ethics/01_CI_CLO
    7  data/courses/SAIA1013_Responsible_AI_and_Ethics/02_Lecture_Slides_Taught
    3  data/courses/SAIA1013_Responsible_AI_and_Ethics/03_Assessments_Assessed
   12  data/courses/SAIA2113_Machine_Learning/02_Lecture_Slides_Taught
    2  data/courses/SAIA2113_Machine_Learning/03_Assessments_Assessed
    1  data/courses/SAIA2143_Data_Wrangling_and_Visualization/01_CI_CLO
   10  data/courses/SAIA2143_Data_Wrangling_and_Visualization/02_Lecture_Slides_Taught
    2  data/courses/SAIA2143_Data_Wrangling_and_Visualization/03_Assessments_Assessed
    2  data/industry_benchmark
Empty evidence folders: ['data/courses/SAIA2113_Machine_Learning/01_CI_CLO']


Opening a file is not enough: a truncated PDF can still open with blank pages, so the integrity check runs first.

In [2]:
check_source_integrity(COURSES_DIR)
check_source_integrity(INDSF_DIR)
print("Source integrity: PASS (PDF end markers and Office archives are complete)")

Source integrity: PASS (PDF end markers and Office archives are complete)


### A3. Inspect two representative sources

In [3]:
import pymupdf as fitz

for path, page_number in ((COURSES_DIR / "SAIA2113_Machine_Learning/02_Lecture_Slides_Taught/Week 13 Clustering.pdf", 9),
                          (INDSF_DIR / "HRD-Corp_IndSF_Artificial_Intelligence.pdf", 7)):
    with fitz.open(path) as document:
        lengths = [len(page.get_text()) for page in document]
        print(path.relative_to(ROOT))
        print("  sha256:", sha256_file(path)[:16], "| pages:", document.page_count,
              "| pages without a text layer:", sum(length == 0 for length in lengths))
        print(f"  page {page_number} sample:", repr(" ".join(document[page_number - 1].get_text().split())[:120]))

data/courses/SAIA2113_Machine_Learning/02_Lecture_Slides_Taught/Week 13 Clustering.pdf
  sha256: de58300fd9d1e40a | pages: 66 | pages without a text layer: 0
  page 9 sample: 'www.utm.my ●Machine learning datasets may have millions of examples and some clustering algorithms may not scale efficie'


data/industry_benchmark/HRD-Corp_IndSF_Artificial_Intelligence.pdf
  sha256: c1e65253310bcd36 | pages: 28 | pages without a text layer: 0
  page 7 sample: 'Investigates enterprise data requirements where there is some complexity and ambiguity. Plans own data modelling and des'


The data inventory is in [`docs/week3_data_inventory.md`](../docs/week3_data_inventory.md).

## Part B: Prepare the data

### B2. Extract both branches

This re-extracts every page with the document cache disabled, so it takes several minutes.

In [4]:
part_b = {kind: extract(kind, **OPTIONS) for kind in KINDS}
for kind, extraction in part_b.items():
    print(kind, "| seconds:", extraction.seconds)
    print(json.dumps(extraction.inspection, indent=1))
print("Known gaps:", part_b[COURSE_KIND].known_gaps)

course_documents | seconds: 401.7
{
 "documents": 38,
 "pages": 1407,
 "documents_by_format": {
  "docx": 1,
  "pdf": 36,
  "pptx": 1
 },
 "pages_by_level": {
  "Assessed": 34,
  "CLO": 7,
  "Taught": 1366
 },
 "level_status": {
  "SAIA1013": {
   "CLO": "available",
   "Taught": "available",
   "Assessed": "available"
  },
  "SAIA2113": {
   "CLO": "folder_empty_or_missing",
   "Taught": "available",
   "Assessed": "available"
  },
  "SAIA2143": {
   "CLO": "available",
   "Taught": "available",
   "Assessed": "available"
  }
 },
 "ocr_pages": 1205,
 "empty_text_pages": 0,
 "indexable_pages": 1376,
 "skip_reasons": {
  "activity_without_explanation": 28,
  "divider": 3
 },
 "redaction_events": {
  "email": 129,
  "office_room": 13,
  "phone": 9
 }
}
indsf_documents | seconds: 35.8
{
 "documents": 2,
 "pages": 49,
 "structured_items": 526,
 "items_by_section": {
  "common_tools": 151,
  "competencies": 138,
  "essential_skills": 78,
  "job_description_tasks": 152,
  "sfia_level_definit

### B3. Compare preparation decisions on the same input

**Decision 1: page-level records or the lab's 500-character splitters?**

In [5]:
course_pages = part_b[COURSE_KIND].comparison_pages
chunking = compare_chunking(course_pages)
print(f"{'approach':<30}{'chunks':>8}{'median':>8}{'max':>8}{'mid-word':>10}{'multi-page':>12}{'page ref':>10}")
for row in chunking:
    print(f"{row['approach']:<30}{row['chunks']:>8}{row['median_chars']:>8}{row['max_chars']:>8}"
          f"{row['ends_mid_word']:>10}{row['spans_multiple_pages']:>12}{row['records_a_page_reference']:>10}")

approach                        chunks  median     max  mid-word  multi-page  page ref
page-level (selected)             1407     427    7714         0           0      1407
fixed-char-500                    1525     500     500       990        1062         0
paragraph-aware-char-500          1525     500     500       990        1062         0


**Decision 2: native text only, or native text plus OCR?** Both use the same pages.

In [6]:
ocr = compare_ocr(course_pages)
for row in ocr:
    print(row)

{'approach': 'native text only', 'pages': 1407, 'total_words': 81750, 'pages_with_no_text': 7, 'pages_under_4_words': 64}
{'approach': 'native + RapidOCR (selected)', 'pages': 1407, 'total_words': 107590, 'pages_with_no_text': 0, 'pages_under_4_words': 7}


### Export and basic output check

In [7]:
manifest = build_lineage_manifest(list(part_b.values()), PROCESSED_DIR, MANIFEST)
for path in sorted(PROCESSED_DIR.glob("*.jsonl")):
    assert path.stat().st_size > 0, "Prepared output is empty."
    print(path.relative_to(ROOT), "| bytes:", path.stat().st_size, "| records:", sum(1 for _ in path.open(encoding="utf-8")))
print("File checks pass; the samples below inspect the content.")

data/processed/prepared_indsf_items.jsonl | bytes: 299523 | records: 526
data/processed/prepared_indsf_text.jsonl | bytes: 109835 | records: 49
data/processed/prepared_text.jsonl | bytes: 2108318 | records: 1407
File checks pass; the samples below inspect the content.


Inspect representative prepared records, not just file sizes:

In [8]:
def show(path, record_id):
    with path.open(encoding="utf-8") as handle:
        row = next(row for row in map(json.loads, handle) if row["record_id"] == record_id)
    print({key: row[key] for key in ("record_id", "source", "page", "processing") if key in row})
    print("  source_sha256:", row["source_sha256"][:16], "| chars:", len(row["text"]))
    print("  text:", repr(row["text"][:120]))

show(PROCESSED_DIR / "prepared_text.jsonl", "SAIA2113__Week 13 Clustering.pdf__p09")
show(PROCESSED_DIR / "prepared_indsf_text.jsonl", "HRD-Corp_IndSF_Artificial_Intelligence.pdf__p07")

{'record_id': 'SAIA2113__Week 13 Clustering.pdf__p09', 'source': 'data/courses/SAIA2113_Machine_Learning/02_Lecture_Slides_Taught/Week 13 Clustering.pdf', 'page': 9, 'processing': 'page-level records; native text + RapidOCR union; PII regex redaction; vision disabled'}
  source_sha256: de58300fd9d1e40a | chars: 571
  text: 'Clustering Algorithms ●Machine learning datasets may have millions of examples and some clustering algorithms may not sc'
{'record_id': 'HRD-Corp_IndSF_Artificial_Intelligence.pdf__p07', 'source': 'data/industry_benchmark/HRD-Corp_IndSF_Artificial_Intelligence.pdf', 'page': 7, 'processing': 'page-level records; native text + RapidOCR union; PII regex redaction; vision disabled; structured B1 item extraction (no embeddings)'}
  source_sha256: c1e65253310bcd36 | chars: 1784
  text: 'AI/ML ENGINEER JOB DESCRIPTION Investigates enterprise data requirements where there is some complexity and ambiguity. P'


### B4. Verify the lineage manifest

This confirms the files exist and match their hashes. It does not prove data quality or permission to share.

In [9]:
print("Lineage records:", verify_manifest(MANIFEST))
record = manifest["records"][0]
print(json.dumps({key: record[key] for key in ("source_id", "source_filename", "pages", "records", "prepared_output", "status")}, indent=1))
print("COMMON LINEAGE VALIDATION: PASS")

Lineage records: 40
{
 "source_id": "SRC01",
 "source_filename": "data/courses/SAIA1013_Responsible_AI_and_Ethics/01_CI_CLO/SAIA1013 Responsible AI and Ethics 2425-1.pdf",
 "pages": 3,
 "records": 3,
 "prepared_output": "data/processed/prepared_text.jsonl",
 "status": "Prepared; quality review pending"
}
COMMON LINEAGE VALIDATION: PASS


## Part C: The V1 data pipeline

### C1. Run every stage for each branch

Source, ingest, inspect, prepare, validate, then record lineage.

In [10]:
first_runs = {kind: execute_pipeline(kind, **OPTIONS) for kind in KINDS}
for kind, run in first_runs.items():
    print("SUCCESS:", kind, "| sources:", len(run["sources"]), "| outputs:", run["validation"]["outputs"],
          "| records:", run["validation"]["output_records"], "| seconds:", run["runtime"]["seconds"])
    print("  residual PII pattern matches:", run["validation"]["residual_pii_pattern_matches"])

SUCCESS: course_documents | sources: 38 | outputs: ['data/pipeline_v1/course_documents/prepared_text.jsonl'] | records: [1407] | seconds: 402.4
  residual PII pattern matches: {'email': 0, 'phone': 0, 'office_room': 0}
SUCCESS: indsf_documents | sources: 2 | outputs: ['data/pipeline_v1/indsf_documents/prepared_indsf_text.jsonl', 'data/pipeline_v1/indsf_documents/prepared_indsf_items.jsonl'] | records: [49, 526] | seconds: 35.1
  residual PII pattern matches: {'email': 0, 'phone': 0, 'office_room': 0}


### Repeatability check

Each branch runs again from the source files with no cache. Timestamps and durations may differ; sources, outputs and mappings may not.

In [11]:
second_runs = {kind: execute_pipeline(kind, **OPTIONS) for kind in KINDS}
results = {kind: compare_runs(first_runs[kind], second_runs[kind]) for kind in KINDS}
for kind, checks in results.items():
    print(kind, checks)
assert all(checks["passed"] for checks in results.values()), "A rerun changed the sources, outputs or mappings."
print("All pipelines passed the repeatability smoke check.")

course_documents {'sources': True, 'validation': True, 'source_output_mapping': True, 'passed': True}
indsf_documents {'sources': True, 'validation': True, 'source_output_mapping': True, 'passed': True}
All pipelines passed the repeatability smoke check.


Part B and Part C should produce the same prepared data, since they use the same configuration:

In [12]:
for kind, run in second_runs.items():
    for output, digest in zip(run["validation"]["outputs"], run["validation"]["output_sha256"]):
        same = sha256_file(PROCESSED_DIR / Path(output).name) == digest
        print(output, "| matches Part B:", same)
        assert same, "Part B and Part C differ; explain this before continuing."
print("Part B and Part C outputs match.")

data/pipeline_v1/course_documents/prepared_text.jsonl | matches Part B: True
data/pipeline_v1/indsf_documents/prepared_indsf_text.jsonl | matches Part B: True
data/pipeline_v1/indsf_documents/prepared_indsf_items.jsonl | matches Part B: True
Part B and Part C outputs match.


### Failure points

These use temporary synthetic files to show each stage stops rather than passing bad data forward.

In [13]:
import shutil, tempfile

def make_pdf(path, pages=20):
    path.parent.mkdir(parents=True, exist_ok=True)
    document = fitz.open()
    for number in range(pages):
        document.new_page().insert_text((72, 72), f"Synthetic page {number + 1} about supervised learning models and evaluation.")
    document.save(path)
    document.close()

NO_OCR = {"enable_ocr": False, "workers": 1, "ocr_threads": 1}
with tempfile.TemporaryDirectory() as temporary:
    project = Path(temporary)
    deck = project / "data/courses/SAIA2113_Machine_Learning/02_Lecture_Slides_Taught/deck.pdf"
    make_pdf(deck)
    good = deck.read_bytes()

    deck.write_bytes(good[:-4096])
    try:
        extract(COURSE_KIND, project, **NO_OCR)
    except PipelineStageError as error:
        print("Failure 1, truncated source: stopped at", error.stage, "|", error)
    else:
        raise AssertionError("A truncated source was not stopped.")

    deck.write_bytes(good)
    extraction = extract(COURSE_KIND, project, **NO_OCR)
    build_lineage_manifest([extraction], project / "data/processed", project / "data/week3_manifest.json", project)
    with (project / "data/processed/prepared_text.jsonl").open("a", encoding="utf-8") as handle:
        handle.write("{}\n")
    try:
        verify_manifest(project / "data/week3_manifest.json", project)
    except PipelineStageError as error:
        print("Failure 2, changed output: stopped at", error.stage, "|", error)
    else:
        raise AssertionError("A changed output was not detected.")

Failure 1, truncated source: stopped at ingest | [ingest] Damaged source files: deck.pdf: no PDF end marker
Failure 2, changed output: stopped at record_lineage | [record_lineage] Checksum mismatch: data/processed/prepared_text.jsonl


### Artefacts

Manifests, IndSF records, the notebook and the docs are committed. Prepared course records stay local because they hold full course text.

In [14]:
import subprocess
for path in sorted([MANIFEST, *PROCESSED_DIR.glob("*"), *PIPELINE_DIR.rglob("*.*")]):
    ignored = subprocess.run(["git", "check-ignore", "-q", str(path)], cwd=ROOT).returncode == 0
    print(f"{'local only' if ignored else 'committable':<12} {path.stat().st_size:>10}  {path.relative_to(ROOT)}")

committable       22116  data/pipeline_v1/course_documents/pipeline_manifest.json
local only      2108318  data/pipeline_v1/course_documents/prepared_text.jsonl
committable        2950  data/pipeline_v1/indsf_documents/pipeline_manifest.json
committable      299523  data/pipeline_v1/indsf_documents/prepared_indsf_items.jsonl
committable      109835  data/pipeline_v1/indsf_documents/prepared_indsf_text.jsonl
committable      299523  data/processed/prepared_indsf_items.jsonl
committable      109835  data/processed/prepared_indsf_text.jsonl
local only      2108318  data/processed/prepared_text.jsonl
committable       40577  data/week3_manifest.json
